# 01 Data sharing agreement — agreement + governance review


In [ ]:
%run 00_env_config


In [ ]:
from datetime import datetime, timezone

from fabricops_kit import (
    draft_business_context,
    draft_governance,
    extract_column_business_context_suggestions,
    extract_governance_suggestions,
    get_reviewed_business_context_rows,
    prepare_business_context_profile_input,
    prepare_governance_input,
    profile_dataframe,
    read_lakehouse_table,
    register_current_notebook,
    review_business_context,
    review_governance,
    setup_notebook,
    write_business_context,
    write_governance,
    write_lakehouse_table,
)


## Layer A: Agreement-level boundary (lightweight inputs only)


In [ ]:
agreement_id = "r002_sales_demo"
agreement_name = "R002 Sales Demo Agreement"
agreement_context = "Demo agreement for FabricOps E2E Test Run 002."
approved_usage = "Use this sample sales order data to test the FabricOps notebook flow."
owner = "FabricOps test owner"
expiry_date = "2026-12-31"
notes = "Agreement-level metadata only. Classification is handled later at table/column level."

env_name = "dev"
save_to_metadata = True
register_notebook_to_metadata = True


In [ ]:
BOOTSTRAP_01 = setup_notebook(
    config=CONFIG,
    env=env_name,
    required_targets=["metadata"],
    notebook_name="01_data_agreement_template",
)

agreement_record = {
    "agreement_id": agreement_id,
    "agreement_name": agreement_name,
    "agreement_context": agreement_context,
    "approved_usage": approved_usage,
    "owner": owner,
    "expiry_date": expiry_date,
    "notes": notes,
    "environment_name": env_name,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

if save_to_metadata:
    agreement_df = spark.createDataFrame([agreement_record])
    write_lakehouse_table(agreement_df, CONFIG, env_name, "metadata", "METADATA_DATA_AGREEMENT", mode="append")
    print("Saved agreement metadata to METADATA_DATA_AGREEMENT.")
else:
    print("Dry run: skipped agreement metadata write.")


## Layer B: Table/column metadata review under the agreement


In [ ]:
dataset_name = "r002_sales_demo"
table_name = "sales_orders"
source_table_identifier = None  # Optional: e.g. "RAW_SALES_ORDERS" for on-demand profiling when metadata is missing.


In [ ]:
profile_rows = []
try:
    profile_df = read_lakehouse_table(CONFIG, env_name, "metadata", "METADATA_PROFILE_ROWS")
    filtered_profile_df = profile_df.filter(
        (profile_df.environment_name == env_name)
        & (profile_df.dataset_name == dataset_name)
        & (profile_df.table_name == table_name)
    )
    profile_rows = [r.asDict(recursive=True) for r in filtered_profile_df.collect()]
except Exception:
    profile_rows = []

if not profile_rows and source_table_identifier:
    source_df = spark.table(source_table_identifier)
    profiled_df = profile_dataframe(source_df, table_name=table_name)
    profile_rows = [r.asDict(recursive=True) for r in profiled_df.collect()]

print(f"Profile rows available for review: {len(profile_rows)}")


In [ ]:
approved_business_context_rows = []
if profile_rows:
    prepared_rows = prepare_business_context_profile_input(
        profile_rows,
        table_name=table_name,
        table_context=agreement_context,
    )
    if prepared_rows:
        prepared_df = spark.createDataFrame(prepared_rows)
        bc_response_df = draft_business_context(prepared_df, prompt_template=CONFIG.ai_prompt_config.business_context_prompt_template)
        bc_suggestions = extract_column_business_context_suggestions(bc_response_df)
        review_business_context(
            bc_suggestions,
            environment_name=env_name,
            dataset_name=dataset_name,
            table_name=table_name,
        )
        print("Business-context review widget launched. Complete review, then run the next save cell.")
else:
    print("No profile evidence found. Skip business-context review or set source_table_identifier.")


In [ ]:
approved_business_context_rows = get_reviewed_business_context_rows("approved")
if save_to_metadata and approved_business_context_rows:
    write_business_context(
        spark,
        rows=approved_business_context_rows,
        metadata_path=CONFIG.path_config.paths[env_name]["metadata"],
        table_name="METADATA_COLUMN_CONTEXT",
        mode="append",
    )
    print(f"Saved approved business-context rows: {len(approved_business_context_rows)}")
else:
    print("No approved business-context rows to save.")


In [ ]:
prepared_governance_rows = prepare_governance_input(
    profile_rows,
    table_name=table_name,
    column_contexts=approved_business_context_rows,
)
if prepared_governance_rows:
    governance_input_df = spark.createDataFrame(prepared_governance_rows)
    governance_response_df = draft_governance(
        governance_input_df,
        prompt=CONFIG.ai_prompt_config.governance_personal_identifier_prompt_template,
    )
    governance_suggestions = extract_governance_suggestions(governance_response_df)
    review_governance(
        governance_suggestions,
        environment_name=env_name,
        dataset_name=dataset_name,
        table_name=table_name,
    )
    print("Governance review widget launched. Complete review, then run the next save cell.")
else:
    print("No governance suggestions available. Ensure business-context approvals exist.")


In [ ]:
if save_to_metadata:
    saved_rows = write_governance(
        spark,
        metadata_path=CONFIG.path_config.paths[env_name]["metadata"],
        agreement_context={"agreement_id": agreement_id, "agreement_context": agreement_context, "approved_usage": approved_usage},
        table_name="METADATA_COLUMN_GOVERNANCE",
        mode="append",
    )
    print(f"Saved governance rows: {len(saved_rows)}")
else:
    print("Dry run: skipped governance write.")


In [ ]:
try:
    dq_rules_df = read_lakehouse_table(CONFIG, env_name, "metadata", "METADATA_DQ_RULES")
    dq_rules_df = dq_rules_df.filter(
        (dq_rules_df.agreement_id == agreement_id)
        & (dq_rules_df.dataset_name == dataset_name)
        & (dq_rules_df.table_name == table_name)
    )
    print(f"DQ rules rows (read-only visibility): {dq_rules_df.count()}")
except Exception:
    print("No METADATA_DQ_RULES table available in this environment.")


In [ ]:
if save_to_metadata and register_notebook_to_metadata:
    register_current_notebook(
        spark,
        metadata_path=CONFIG.path_config.paths[env_name]["metadata"],
        agreement_id=agreement_id,
        notebook_type="01_data_sharing_agreement",
        environment_name=env_name,
        dataset_name=dataset_name,
        table_name=table_name,
    )
    print("Registered notebook in METADATA_NOTEBOOK_REGISTRY.")
